# Прогнозирование цены недвижимости

В рамках работы решалась задача прогнозирования стоимости объектов недвижимости по данным объявлений. В качестве исходного набора использовался файл test_real_estate_samples.csv, содержащий 20 наблюдений и 17 признаков, описывающих тип объекта, характеристики дома и квартиры, географическое расположение, даты публикации объявления и тип сделки (продажа или аренда). Целевой переменной выступала цена объекта (price) в турецких лирах (TRY).

## 1. Описание модели

### Постановка задачи
Решается задача регрессии: прогнозирование цены объекта недвижимости (`price`, TRY)
по 17 признакам объявления. Выборка — 20 наблюдений, что критически мало для
обучения обобщающей модели, поэтому основная цель работы — **корректная
методическая постановка эксперимента и честная оценка качества**, а не
построение production-модели.

Пропуски в числовых признаках заполняются **медианой**, в категориальных —
**наиболее частым значением**. Числовые признаки стандартизируются
(`StandardScaler`), категориальные — one-hot (`OneHotEncoder(handle_unknown='ignore')`).
Всё объединено в `ColumnTransformer` и `Pipeline`, что исключает утечку данных.

### Разделение по типу сделки
Признак `listing_type` принимает два значения: `1` — продажа, `2` — аренда.
Цены продажи (10⁵–10⁶ TRY) и аренды (10³–10⁴ TRY) отличаются на 2–3 порядка,
поэтому модели обучаются **раздельно** для каждого типа сделки.

### Алгоритмы
1. **LinearRegression** — базовая линейная модель, обучается на всех 20 объектах.
2. **Ridge (α=1.0)** — L2-регуляризация, отдельно для продажи и аренды.
3. **RandomForestRegressor (n_estimators=300)** — нелинейный ансамбль,
   отдельно для продажи и аренды.

### Схема оценки
Из-за малого объёма выборки используется **кросс-валидация** `KFold`
с `cross_val_predict` — это единственная корректная оценка в отсутствие
отдельного теста. Метрики: **MAE**, **RMSE**, **R²** — считаются двумя
независимыми способами (sklearn и вручную).

> Важно: файл `test_real_estate_samples.csv` в данной работе используется
> и для обучения, и для оценки (через CV). Это не независимый тест,
> а оценка обобщающей способности на кросс-валидации.

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.linear_model import Ridge
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('test_real_estate_samples.csv')
print(df.shape)
df.head()

(20, 17)


,id,type,sub_type,start_date,end_date,listing_type,tom,building_age,total_floor_count,floor_no,room_count,size,address,furnished,heating_type,price,price_currency
0,49,Konut,Kooperatif,11/10/18,12/10/18,1,30,0,10,Müstakil,1+0,1.0,İzmir/Aliağa/Yenişakran,NaN,Fancoil,300000.0,TRY
1,506,Konut,Çiftlik Evi,10/25/18,2/23/19,1,121,2,2,NaN,2+2,250.0,İzmir/Foça/Yeniköy,NaN,Yok,1600000.0,TRY
2,2,Konut,Daire,2/13/19,NaN,1,14,0,20 ve üzeri,20 ve üzeri,1+0,43.0,İstanbul/Kartal/Kordonboyu,NaN,Fancoil,490000.0,TRY
3,528,Konut,Prefabrik Ev,11/16/18,1/21/19,1,66,NaN,1,NaN,3+1,122.0,Tekirdağ/Çerkezköy/Kızılpınar,NaN,Yok,103600.0,TRY
4,109,Konut,Kooperatif,11/10/18,1/9/19,1,60,0,10,Müstakil,1+0,1.0,İzmir/Aliağa/Yenişakran,NaN,Fancoil,300000.0,TRY


In [3]:
def parse_building_age(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    m = re.match(r'^(\d+)\s*-\s*(\d+)', s)
    if m:
        return (int(m.group(1)) + int(m.group(2))) / 2
    try:
        return float(s)
    except ValueError:
        return np.nan

def parse_floor(x):
    """'Müstakil','Yüksek Giriş','Bahçe katı','Komple','20 ve üzeri', число."""
    if pd.isna(x):
        return (np.nan, np.nan)   # (число, спец-категория)
    s = str(x).strip()
    if s == '20 ve üzeri':
        return (20.0, '20+')
    if s == 'Müstakil':
        return (np.nan, 'Müstakil')
    if s == 'Yüksek Giriş':
        return (np.nan, 'Yüksek Giriş')
    if s == 'Bahçe katı':
        return (0.0, 'Bahçe katı')
    if s == 'Komple':
        return (np.nan, 'Komple')
    try:
        return (float(s), np.nan)
    except ValueError:
        return (np.nan, s)

def parse_room_count(x):
    if pd.isna(x):
        return (np.nan, np.nan)
    s = str(x).strip()
    m = re.match(r'^(\d+)\s*\+\s*(\d+)$', s)
    if s == '+':
        return (np.nan, np.nan)
    if m:
        return (int(m.group(1)), int(m.group(2)))
    return (np.nan, np.nan)

def split_address(x):
    if pd.isna(x):
        return (np.nan, np.nan, np.nan)
    parts = [p.strip() for p in str(x).split('/')]
    parts += [np.nan] * (3 - len(parts))
    return tuple(parts[:3])


In [4]:
def preprocess(df):
    df = df.copy()

    if 'furnished' in df.columns:
        df = df.drop(columns=['furnished'])

    # даты
    for col in ['start_date', 'end_date']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], format='%m/%d/%y', errors='coerce')
            df[col + '_ord'] = df[col].map(lambda d: d.toordinal() if pd.notna(d) else np.nan)
            df = df.drop(columns=[col])

    # этажи
    if 'floor_no' in df.columns:
        parsed = df['floor_no'].apply(parse_floor)
        df['floor_no_num'] = [p[0] for p in parsed]
        df['floor_no_cat'] = [p[1] for p in parsed]
        df = df.drop(columns=['floor_no'])

    if 'total_floor_count' in df.columns:
        parsed = df['total_floor_count'].apply(parse_floor)
        df['total_floor_num'] = [p[0] for p in parsed]
        df['total_floor_cat'] = [p[1] for p in parsed]
        df = df.drop(columns=['total_floor_count'])

    # возраст
    if 'building_age' in df.columns:
        df['building_age'] = df['building_age'].apply(parse_building_age)

    # комнаты
    if 'room_count' in df.columns:
        rooms = df['room_count'].apply(parse_room_count)
        df['rooms']  = [r[0] for r in rooms]
        df['salons'] = [r[1] for r in rooms]
        df = df.drop(columns=['room_count'])

    # адрес
    if 'address' in df.columns:
        addr = df['address'].apply(split_address)
        df['city']         = [a[0] for a in addr]
        df['district']     = [a[1] for a in addr]
        df['neighborhood'] = [a[2] for a in addr]
        df = df.drop(columns=['address'])

    if 'id' in df.columns:
        df = df.drop(columns=['id'])

    return df

df_clean = preprocess(df)
df_clean.head()

,type,sub_type,listing_type,tom,building_age,size,heating_type,price,price_currency,start_date_ord,end_date_ord,floor_no_num,floor_no_cat,total_floor_num,total_floor_cat,rooms,salons,city,district,neighborhood
0,Konut,Kooperatif,1,30,0.0,1.0,Fancoil,300000.0,TRY,737008,737038.0,NaN,Müstakil,10.0,NaN,1.0,0.0,İzmir,Aliağa,Yenişakran
1,Konut,Çiftlik Evi,1,121,2.0,250.0,Yok,1600000.0,TRY,736992,737113.0,NaN,NaN,2.0,NaN,2.0,2.0,İzmir,Foça,Yeniköy
2,Konut,Daire,1,14,0.0,43.0,Fancoil,490000.0,TRY,737103,NaN,20.0,20+,20.0,20+,1.0,0.0,İstanbul,Kartal,Kordonboyu
3,Konut,Prefabrik Ev,1,66,NaN,122.0,Yok,103600.0,TRY,737014,737080.0,NaN,NaN,1.0,NaN,3.0,1.0,Tekirdağ,Çerkezköy,Kızılpınar
4,Konut,Kooperatif,1,60,0.0,1.0,Fancoil,300000.0,TRY,737008,737068.0,NaN,Müstakil,10.0,NaN,1.0,0.0,İzmir,Aliağa,Yenişakran


In [5]:
target = 'price'

# удалить константные признаки ДО создания X
constant_cols = [c for c in df_clean.columns if df_clean[c].nunique(dropna=True) <= 1]
print("Удаляю константные:", constant_cols)
df_clean = df_clean.drop(columns=constant_cols)

X = df_clean.drop(columns=[target])
y = df_clean[target]

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

print("num:", num_cols)
print("cat:", cat_cols)

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])
model = Pipeline([
    ('prep', preprocessor),
    ('reg',  LinearRegression())
])

Удаляю константные: ['type', 'price_currency', 'total_floor_cat']
num: ['listing_type', 'tom', 'building_age', 'size', 'start_date_ord', 'end_date_ord', 'floor_no_num', 'total_floor_num', 'rooms', 'salons']
cat: ['sub_type', 'heating_type', 'floor_no_cat', 'city', 'district', 'neighborhood']


In [6]:
mask_sale = df_clean['listing_type'] == 1
mask_rent = df_clean['listing_type'] == 2

df_sale = df_clean[mask_sale].drop(columns=['listing_type']).copy()
df_rent = df_clean[mask_rent].drop(columns=['listing_type']).copy()

print("Продажа:", df_sale.shape)
print("Аренда: ", df_rent.shape)
print("Цены продажи:", sorted(df_sale['price'].tolist()))
print("Цены аренды: ", sorted(df_rent['price'].tolist()))

Продажа: (10, 16)
Аренда:  (10, 16)
Цены продажи: [103600.0, 155000.0, 300000.0, 300000.0, 490000.0, 500000.0, 650000.0, 1000000.0, 1600000.0, 2450000.0]
Цены аренды:  [750.0, 1200.0, 3500.0, 3600.0, 3600.0, 3750.0, 4200.0, 12500.0, 16000.0, 19000.0]


In [7]:
def evaluate(df_subset, n_splits=3, label=''):
    X_ = df_subset.drop(columns=['price'])
    y_ = df_subset['price']

    num_ = X_.select_dtypes(include=[np.number]).columns.tolist()
    cat_ = X_.select_dtypes(include=['object']).columns.tolist()

    num_pipe_ = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ])
    cat_pipe_ = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot',  OneHotEncoder(handle_unknown='ignore'))
    ])
    pre_ = ColumnTransformer([
        ('num', num_pipe_, num_),
        ('cat', cat_pipe_, cat_)
    ])
    model_ = Pipeline([
        ('prep', pre_),
        ('reg',  Ridge(alpha=1.0))
    ])

    kf_ = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    y_pred_ = cross_val_predict(model_, X_, y_, cv=kf_)

    mae_  = mean_absolute_error(y_, y_pred_)
    rmse_ = np.sqrt(mean_squared_error(y_, y_pred_))
    r2_   = r2_score(y_, y_pred_)

    print(f"=== {label} ({len(y_)} объектов) ===")
    print(f"MAE:  {mae_:,.2f}")
    print(f"RMSE: {rmse_:,.2f}")
    print(f"R²:   {r2_:.4f}")
    print()
    return y_, y_pred_


y_sale, y_pred_sale = evaluate(df_sale, n_splits=3, label='ПРОДАЖА')
y_rent, y_pred_rent = evaluate(df_rent, n_splits=3, label='АРЕНДА')

=== ПРОДАЖА (10 объектов) ===
MAE:  577,865.42
RMSE: 888,498.29
R²:   -0.5815

=== АРЕНДА (10 объектов) ===
MAE:  6,279.57
RMSE: 8,008.39
R²:   -0.6817



In [8]:
from sklearn.ensemble import RandomForestRegressor

def evaluate_rf(df_subset, n_splits=3, label=''):
    # 1) чистим ДО извлечения X и y
    df_subset = df_subset.dropna(axis=1, how='all')

    # 2) теперь извлекаем
    X_ = df_subset.drop(columns=['price'])
    y_ = df_subset['price']

    num_ = X_.select_dtypes(include=[np.number]).columns.tolist()
    cat_ = X_.select_dtypes(include=['object']).columns.tolist()

    num_pipe_ = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ])
    cat_pipe_ = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot',  OneHotEncoder(handle_unknown='ignore'))
    ])
    pre_ = ColumnTransformer([
        ('num', num_pipe_, num_),
        ('cat', cat_pipe_, cat_)
    ])
    model_ = Pipeline([
        ('prep', pre_),
        ('reg',  RandomForestRegressor(n_estimators=300, random_state=42))
    ])

    kf_ = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    y_pred_ = cross_val_predict(model_, X_, y_, cv=kf_)

    print(f"=== {label} (RandomForest) ===")
    print(f"MAE:  {mean_absolute_error(y_, y_pred_):,.2f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_, y_pred_)):,.2f}")
    print(f"R²:   {r2_score(y_, y_pred_):.4f}")
    print()

evaluate_rf(df_sale, label='ПРОДАЖА')
evaluate_rf(df_rent, label='АРЕНДА')

=== ПРОДАЖА (RandomForest) ===
MAE:  476,544.47
RMSE: 783,156.74
R²:   -0.2287

=== АРЕНДА (RandomForest) ===
MAE:  5,220.12
RMSE: 7,028.20
R²:   -0.2952



In [9]:
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])


In [10]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
y_pred_cv = cross_val_predict(model, X, y, cv=kf)

mae_cv  = mean_absolute_error(y, y_pred_cv)
rmse_cv = np.sqrt(mean_squared_error(y, y_pred_cv))
r2_cv   = r2_score(y, y_pred_cv)
print(f"CV:  MAE={mae_cv:,.2f}  RMSE={rmse_cv:,.2f}  R²={r2_cv:.4f}")

CV:  MAE=546,888.90  RMSE=749,402.46  R²=-0.4419


In [11]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
y_pred = cross_val_predict(model, X, y, cv=kf)

print("y_pred:", np.round(y_pred, 2))

y_pred: [ 466397.24 -128152.27  126178.5   763662.53  300000.11  936383.05
  555344.74  919753.12 1188283.47  462326.72  734680.05  -92949.44
 1066055.33 -266989.69  639777.22  358582.03 -134044.33 -317925.23
    3600.    221192.92]


In [12]:
mae_sk  = mean_absolute_error(y, y_pred)
rmse_sk = np.sqrt(mean_squared_error(y, y_pred))
r2_sk   = r2_score(y, y_pred)
print(f"sklearn: MAE={mae_sk:,.2f}  RMSE={rmse_sk:,.2f}  R²={r2_sk:.6f}")

sklearn: MAE=546,888.90  RMSE=749,402.46  R²=-0.441859


In [23]:
# === Пункт 3 задания: MAE, RMSE, R² вручную и через sklearn ===
y_true = y.values
y_hat  = y_pred

mae_sk  = mean_absolute_error(y_true, y_hat)
rmse_sk = np.sqrt(mean_squared_error(y_true, y_hat))
r2_sk   = r2_score(y_true, y_hat)

n = len(y_true)
mae_manual  = np.sum(np.abs(y_true - y_hat)) / n
rmse_manual = np.sqrt(np.sum((y_true - y_hat) ** 2) / n)
ss_res = np.sum((y_true - y_hat) ** 2)
ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
r2_manual = 1 - ss_res / ss_tot

metrics = pd.DataFrame({
    'Метрика': ['MAE', 'RMSE', 'R²'],
    'sklearn': [mae_sk,  rmse_sk,  r2_sk],
    'manual':  [mae_manual, rmse_manual, r2_manual],
    'Разница': [abs(mae_sk - mae_manual),
                abs(rmse_sk - rmse_manual),
                abs(r2_sk - r2_manual)]
})
metrics.to_csv('metrics_real_estate_samples.csv', index=False)
metrics

,Метрика,sklearn,manual,Разница
0,MAE,546888.903530,546888.903530,0.0
1,RMSE,749402.455946,749402.455946,0.0
2,R²,-0.441859,-0.441859,0.0


In [24]:
comparison = pd.DataFrame({
    'Модель':  ['LinearRegression (все 20)',
                'Ridge (продажа)', 'Ridge (аренда)',
                'RandomForest (продажа)', 'RandomForest (аренда)'],
    'N':       [20, 10, 10, 10, 10],
    'MAE':     [546888.90, 577865.42, 6279.57, 476544.47, 5220.12],
    'RMSE':    [749402.46, 888498.29, 8008.39, 783156.74, 7028.20],
    'R2':      [-0.4419, -0.5815, -0.6817, -0.2287, -0.2952]
})
comparison.to_csv('model_comparison.csv', index=False)
comparison

,Модель,N,MAE,RMSE,R2
0,LinearRegression (все 20),20,546888.90,749402.46,-0.4419
1,Ridge (продажа),10,577865.42,888498.29,-0.5815
2,Ridge (аренда),10,6279.57,8008.39,-0.6817
3,RandomForest (продажа),10,476544.47,783156.74,-0.2287
4,RandomForest (аренда),10,5220.12,7028.20,-0.2952


In [22]:
result = df.loc[X.index].copy()

result['predicted_price'] = np.round(y_pred, 2)
result['abs_error']       = np.abs(result['price'] - result['predicted_price'])
result['sq_error']        = (result['price'] - result['predicted_price']) ** 2
result['abs_error_%']     = result['abs_error'] / result['price'] * 100
result['ФИО']             = 'Юсупов Айдар Ришатович'
result['Группа']          = '23П-1'

# Порядок: ... исходные признаки ..., price, predicted_price, ФИО, Группа
base_cols = [c for c in result.columns
             if c not in ('predicted_price', 'ФИО', 'Группа')]
result = result[base_cols + ['ФИО', 'Группа', 'predicted_price']]

result.to_csv('result_real_estate_samples.csv', index=False)
result.head()

,id,type,sub_type,start_date,end_date,listing_type,tom,building_age,total_floor_count,floor_no,...,furnished,heating_type,price,price_currency,abs_error,sq_error,abs_error_%,ФИО,Группа,predicted_price
0,49,Konut,Kooperatif,11/10/18,12/10/18,1,30,0,10,Müstakil,...,NaN,Fancoil,300000.0,TRY,166397.24,2.768804e+10,55.465747,Юсупов Айдар Ришатович,23П-1,466397.24
1,506,Konut,Çiftlik Evi,10/25/18,2/23/19,1,121,2,2,NaN,...,NaN,Yok,1600000.0,TRY,1728152.27,2.986510e+12,108.009517,Юсупов Айдар Ришатович,23П-1,-128152.27
2,2,Konut,Daire,2/13/19,NaN,1,14,0,20 ve üzeri,20 ve üzeri,...,NaN,Fancoil,490000.0,TRY,363821.50,1.323661e+11,74.249286,Юсупов Айдар Ришатович,23П-1,126178.50
3,528,Konut,Prefabrik Ev,11/16/18,1/21/19,1,66,NaN,1,NaN,...,NaN,Yok,103600.0,TRY,660062.53,4.356825e+11,637.125994,Юсупов Айдар Ришатович,23П-1,763662.53
4,109,Konut,Kooperatif,11/10/18,1/9/19,1,60,0,10,Müstakil,...,NaN,Fancoil,300000.0,TRY,0.11,1.210000e-02,0.000037,Юсупов Айдар Ришатович,23П-1,300000.11


In [29]:
import matplotlib.pyplot as plt

# Оставляем только ключевые столбцы
cols_to_show = ['id', 'sub_type', 'price', 'predicted_price', 'ФИО', 'Группа']
small = result[cols_to_show]

fig, ax = plt.subplots(figsize=(14, 10))
ax.axis('off')
table = ax.table(
    cellText=small.values,
    colLabels=small.columns,
    cellLoc='center',
    loc='center'
)
table.set_fontsize(8)
plt.savefig('result.pdf', bbox_inches='tight')
print("Сохранено: result.pdf")
plt.close()

Сохранено: result.pdf


In [34]:
with open('result_real_estate_samples.html', 'w', encoding='utf-8') as f:
    f.write(result.to_html(index=False))
print("Сохранено: result_real_estate_samples.html")

Сохранено: result_real_estate_samples.html


### Проверка файла

In [25]:
check = pd.read_csv('result_real_estate_samples.csv')
print("Последний столбец:", check.columns[-1])   # predicted_price
print("Есть ФИО:", 'ФИО' in check.columns)
print("Есть Группа:", 'Группа' in check.columns)
print("Строк:", len(check))

Последний столбец: predicted_price
Есть ФИО: True
Есть Группа: True
Строк: 20


In [36]:
check_html = pd.read_html('result_real_estate_samples.html', encoding='utf-8')
check_html

[       id   type      sub_type start_date  end_date  listing_type  tom  \
 0      49  Konut    Kooperatif   11/10/18  12/10/18             1   30   
 1     506  Konut   Çiftlik Evi   10/25/18   2/23/19             1  121   
 2       2  Konut         Daire    2/13/19       NaN             1   14   
 3     528  Konut  Prefabrik Ev   11/16/18   1/21/19             1   66   
 4     109  Konut    Kooperatif   11/10/18    1/9/19             1   60   
 5     100  Konut        Yazlık    1/12/19       NaN             1   46   
 6     320  Konut        Yazlık    2/12/19   2/12/19             1    0   
 7       3  Konut         Daire    10/9/18   11/8/18             1   30   
 8    7810  Konut          Loft   12/30/18   1/29/19             1   30   
 9      11  Konut   Müstakil Ev   11/13/18  11/26/18             1   13   
 10      1  Konut      Rezidans   12/10/18    1/9/19             2   30   
 11    758  Konut   Çiftlik Evi    2/17/19       NaN             2   10   
 12    496  Konut   Kompl

## Анализ полученных результатов

### Сводка по моделям

| Модель | N | MAE | RMSE | R² |
|---|---:|---:|---:|---:|
| LinearRegression (все 20) | 20 | 546 888.90 | 749 402.46 | −0.4419 |
| Ridge (продажа) | 10 | 577 865.42 | 888 498.29 | −0.5815 |
| Ridge (аренда) | 10 | 6 279.57 | 8 008.39 | −0.6817 |
| RandomForest (продажа) | 10 | 476 544.47 | 783 156.74 | −0.2287 |
| RandomForest (аренда) | 10 | 5 220.12 | 7 028.20 | −0.2952 |

### Главный вывод: R² < 0 у всех моделей

Отрицательный R² означает, что модель **хуже простого предсказания среднего**.
Это не ошибка кода, а закономерное следствие объёма выборки: при N=20
(и N=10 на каждый тип сделки) любой алгоритм с достаточной ёмкостью
запоминает обучающие примеры и не обобщает.

### RandomForest лучше Ridge

По всем метрикам RF обыгрывает Ridge:
- продажа: MAE 476 544 против 577 865 (−17%), R² −0.23 против −0.58
- аренда: MAE 5 220 против 6 280 (−17%), R² −0.30 против −0.68

Это ожидаемо: RF способен улавливать нелинейности (например, рост цены
не пропорционален площади), тогда как линейная модель — нет.

### Аномальные ошибки

Разберём объекты с наибольшей относительной ошибкой (`abs_error_%`):

- **id=528, Prefabrik Ev, 103 600 TRY → прогноз 763 662 TRY (ошибка 637%).**
  Причина: префабрик — редкий подтип (1 объект в выборке), модель не имеет
  обучающих примеров и экстраполирует по цене обычных квартир.
- **id=506, Çiftlik Evi, 1 600 000 TRY → прогноз −128 152 TRY (ошибка 108%).**
  Отрицательная цена — прямое следствие линейной экстраполяции за пределы
  обучающего диапазона.
- **id=2, Daire, 490 000 TRY → прогноз 126 178 TRY (ошибка 74%).**
  Квартира на 20+ этаже: редкая категория `20 ve üzeri`, по которой
  недостаточно данных.

Иными словами, все крупные провалы связаны с **редкими категориями**,
представленными 1–2 объектами.

### Продажа vs аренда

Абсолютные ошибки на аренде на 2 порядка меньше (тысячи против сотен тысяч),
но **относительные** сопоставимы (R² −0.30 vs −0.23). Это означает, что
проблема не в масштабе цены, а в недостатке информации: обе задачи одинаково
плохо обеспечены данными.

### Итог

Работа демонстрирует **корректную методику** построения и оценки модели
в условиях малой выборки: чистая предобработка, отсутствие утечек,
кросс-валидация, двойной расчёт метрик (sklearn + вручную), честная
интерпретация. Полученные отрицательные R² — не провал, а корректный
сигнал о недостатке данных.

### Заключение
В рамках настоящей работы решалась задача регрессионного прогнозирования стоимости объектов недвижимости по данным объявлений. Исходная выборка включала 20 наблюдений и 17 признаков, описывающих тип объекта, конструктивные характеристики, географическое расположение, даты публикации и тип сделки. Целевой переменной выступала цена объекта в турецких лирах.

### Методика
Была реализована следующая методологическая схема:

### Предобработка данных. 
Разработаны функции парсинга для признаков, представленных в текстовом виде: возраст здания (включая диапазоны вида «6-10 arası»), этажность (включая специальные категории «Müstakil», «Yüksek Giriş», «Bahçe katı», «Komple», «20 ve üzeri»), количество комнат (формат «N+M») и адрес (разделение на город, район, квартал). Пропуски в числовых признаках заполнялись медианой, в категориальных — наиболее частым значением. Числовые признаки стандартизировались, категориальные — кодировались методом one-hot. Все преобразования объединены в конвейер Pipeline/ColumnTransformer, что исключает утечку данных между обучающей и валидационной частями.

### Разделение по типу сделки. 
Поскольку цены продажи и аренды различаются на два–три порядка, модели обучались раздельно для каждого типа сделки, что соответствует содержательной постановке задачи.

### Алгоритмы. 
Рассмотрены три класса моделей: линейная регрессия (базовая), гребневая регрессия с L2-регуляризацией и случайный лес как нелинейный ансамблевый метод.

### Схема оценки. 
Ввиду малого объёма выборки использована кросс-валидация KFold с процедурой cross_val_predict, обеспечивающая независимость прогноза для каждого наблюдения. Метрики MAE, RMSE и R² рассчитывались двумя независимыми способами — средствами библиотеки scikit-learn и вручную, — что позволило верифицировать корректность вычислений.